# Inverse reconstruction — hidden fluid mechanics

This is the headline. I give the network **only sparse, noisy velocity measurements** and ask it to recover:

1. the **full velocity field**,
2. the **pressure field it never saw**, and
3. the **unknown viscosity $\nu$**.

There is no pressure data and no boundary/initial conditions — pressure and the interior field come from the physics, and $\nu$ is a single learnable parameter identified from the noisy data.

## Setup

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from src.config import Config
from src.model import PINN
from src import data, pde, train, inverse, utils, viz

cfg = Config(n_layers=6, n_units=64, n_collocation=15000,
             adam_iters=3000, lbfgs_iters=2000, log_every=250)
utils.set_seed(cfg.seed)
device = utils.get_device(cfg.prefer_cuda)
generator = torch.Generator(device=device).manual_seed(cfg.seed)
N_MEAS, NOISE, NU_INIT = 4000, 0.03, 0.05
print(f'true nu = {cfg.nu}')

## The measurements

I sample 4000 scattered $(x,y,t)$ points across the whole domain and all time slices, record the exact velocity there, and add **3% Gaussian noise**. Pressure is never sampled. Sampling across time matters: the Taylor–Green decay rate is $\exp(-2\nu t)$, so multiple time slices are what make $\nu$ identifiable from velocity alone.

In [ ]:
meas, coll = train.make_inverse_data(cfg, N_MEAS, NOISE, device, generator)
xm, ym, tm, um, vm = meas
print(f'{xm.shape[0]} scattered noisy velocity samples; pressure hidden')

## Learnable viscosity

$\nu$ is a single parameter kept positive with a softplus. I start it **5× too large** (0.05 vs the true 0.01) so the recovery is a real result, not a lucky initialization.

In [ ]:
model = PINN(cfg).to(device)
nu_module = inverse.LearnableNu(init=NU_INIT).to(device)
print(f'nu initialized at {nu_module.value:.4f} (true {cfg.nu})')

## Train

Data misfit on the noisy velocity plus the PDE residual. $\nu$ is optimized jointly with the network weights, and I keep its whole trajectory for the convergence plot.

In [ ]:
history, nu_traj = train.train_inverse(model, nu_module, cfg, meas, coll)
nu_est = nu_module.value
print(f'recovered nu = {nu_est:.5f}  vs true {cfg.nu}  '
      f'(error {inverse.nu_error(nu_est, cfg.nu):.2%})')

## How well was the field reconstructed?

Relative $L^2$ against the exact fields — including pressure, which the network never saw.

In [ ]:
for tt in [cfg.t_min, 0.5*cfg.t_max, cfg.t_max]:
    gx, gy, gt, shape = data.grid_eval_points(cfg, t=tt, n=100, device=device)
    for z in (gx, gy, gt): z.requires_grad_(True)
    up, vp, pp = pde.predict_uvp(model, gx, gy, gt)
    ut, vt, pt = data.tgv_fields(gx, gy, gt, cfg.nu)
    eu = utils.relative_l2(up, ut); ev = utils.relative_l2(vp, vt)
    ep = utils.relative_l2(pp-pp.mean(), pt-pt.mean())
    print(f't={tt:g}:  u={eu:.2%}  v={ev:.2%}  p={ep:.2%}')

## The story in three pictures

**Viscosity is identified from noisy data** — wrong start, Adam drifts it down, L-BFGS locks it onto the truth:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg
plt.figure(figsize=(7,4.5)); plt.imshow(mpimg.imread('results/figures/inverse_nu.png'))
plt.axis('off'); plt.show()

**Pressure is recovered without ever being measured:**

In [ ]:
plt.figure(figsize=(15,4.2)); plt.imshow(mpimg.imread('results/figures/inverse_p_t1.png'))
plt.axis('off'); plt.show()

**Vorticity** $\omega = v_x - u_y$ — a derived quantity, so a good check that the reconstruction is physically smooth:

In [ ]:
plt.figure(figsize=(15,4.2)); plt.imshow(mpimg.imread('results/figures/inverse_vorticity_t1.png'))
plt.axis('off'); plt.show()

Everything here is reproduced by `python run_inverse.py`, which writes the recovered $\nu$ and all error metrics to `results/metrics.json`.